# Paper Figures — Invariant Bench

This notebook:
1. Gathers all `experiment1_pcl_results.csv` and `experiment2_accuracy_results.csv` from result folders into a single merged CSV.
2. For each experiment configuration, computes the **empirical** feature-switch point (from accuracy-drop crossover) and the **theoretical** MDL-predicted switch point.
3. Produces a scatter plot: empirical switch N vs. theoretical switch N.

In [ ]:
import sys, warnings
from pathlib import Path
%load_ext autoreload
%autoreload 2

## INPUT NEED : Provide your experiment path

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Provide your experiment folder
# ═══════════════════════════════════════════════════════════════════════
experiment_path = "results/20260221-184131"

In [ ]:

PROJECT_ROOT = Path.cwd().resolve()

RESULTS_ROOT = PROJECT_ROOT / experiment_path
OUTPUT_DIR = RESULTS_ROOT / 'plot'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MERGED_EXP1_CSV = OUTPUT_DIR / 'merged_experiment1_pcl_results.csv'
MERGED_EXP2_CSV = OUTPUT_DIR / 'merged_experiment2_accuracy_results.csv'

SAVE_FIGURES = True  # Set to True to export plots as PNG and PDF

warnings.filterwarnings('ignore', category=FutureWarning)

# ═══════════════════════════════════════════════════════════════════════
# ██  TUNABLE CONFIGURATION  ██
# ═══════════════════════════════════════════════════════════════════════

# ── Bayes model ──────────────────────────────────────────────────────
SKIP_BAYES_GLOBAL = False           # When True, ignore Bayes model everywhere
                                    # (aggregated scatter plots AND individual plots)

# ── Envelope / Theory ────────────────────────────────────────────────
INCLUDE_BAYES_IN_ENVELOPE = False   # Include Bayes model in the MDL envelope
BAYES_ASYMPTOTIC_ONLY = True      # When True, include only the asymptotic Bayes
                                   # line (K(p) + N·loss); when False, also include
                                   # all threshold/interpolated Bayes intermediate
                                   # models. Only has effect if INCLUDE_BAYES_IN_ENVELOPE=True.

# ── K(p) convergence cutoff ─────────────────────────────────────────
KP_ABS_DELTA_THRESHOLD = -1e-5    # Absolute delta threshold for K(p) convergence
                                   # detection.  Lower → keep more tail; higher → cut earlier.

# ── Axis limits (shared across all scatter plots) ────────────────────
AXIS_LO = 10
AXIS_HI = 100_000

# ── Minimum N threshold for transition points ───────────────────────
N_MIN = 20                          # Filter out N_empirical and N_theoretical < N_MIN

# ── Folder-name prefix filters (one per plot) ────────────────────────
# Each plot only includes experiments whose folder starts with this prefix.
# Set to None or '' to disable filtering for that plot.
PLOT1_FOLDER_PREFIX = 'config1_'    # Plot 1: effect of flip_prob
PLOT2_FOLDER_PREFIX = 'config2_'    # Plot 2: effect of spurious mechanism noise
PLOT3_FOLDER_PREFIX = 'config3_'    # Plot 3: effect of bank size

# ── Plot 1: Effect of flip_prob ──────────────────────────────────────
PLOT1_S2_ENV_NOISINESS = 0.0       # Fix env_noisiness for Setting 2
PLOT1_S2_BANK_SIZE = 50            # Fix bank size for Setting 2
PLOT1_DIGITS_PER_CLASS_S2 = None      # Filter digits_per_class for Setting 2 (None = all)

# ── Plot 2: Effect of spurious mechanism noise ───────────────────────
PLOT2_FLIP_PROB_S1 = 0.0          # Fix flip_prob for Setting 1
PLOT2_DIGITS_PER_CLASS = None         # Filter digits_per_class (None = all)

# ── Plot 3: Effect of spurious mechanism complexity ──────────────────
PLOT3_FLIP_PROB = 0.1
PLOT3_ENV_NOISE = 0.0
PLOT3_DIGITS_PER_CLASS = None        # Filter digits_per_class (None = all)

# ── Envelope model counts ────────────────────────────────────────────
NB_THRESHOLD_MODELS = 0            # Accuracy-snapped threshold models (0 to disable)
NB_INTERPOLATED_MODELS = 50       # Loss-interpolated intermediate models

THRESHOLD_METRIC = 'k(p)'  #'mean_original_acc'
  # any PCL column; "acc" → >=, else → <=



def _filter_by_folder_prefix(df, prefix, label=''):
    """Filter a DataFrame to rows whose 'folder' starts with *prefix*."""
    if not prefix:
        return df
    if 'folder' not in df.columns:
        print(f"WARNING: {label} has no 'folder' column — no filtering applied")
        return df
    before = len(df)
    df = df[df['folder'].str.startswith(prefix)]
    print(f"[{label}] Filtered folder prefix '{prefix}': Kept {len(df)}/{before} rows")
    return df
def _filter_by_digits_per_class(df, digits_per_class, label=''):
    """Filter a DataFrame by digits_per_class if specified (not None)."""
    if digits_per_class is None:
        return df
    if 'digits_per_class' not in df.columns:
        print(f"WARNING: {label} has no 'digits_per_class' column — no filtering applied")
        return df
    before = len(df)
    df = df[df['digits_per_class'] == digits_per_class]
    print(f"[{label}] Filtered digits_per_class == {digits_per_class}: Kept {len(df)}/{before} rows")
    return df

## 1. Gather & Merge All CSVs

Set `REBUILD_MERGED_CSV = True` to re-scan all result folders and rebuild the merged CSVs.  
Set to `False` to load the previously saved merged CSVs.

In [ ]:
import pandas as pd
import numpy as np
from glob import glob

# ── Toggle: rebuild from scratch or load from disk ──────────────────
REBUILD_MERGED_CSV = True

def _resolve_seed_folder(experiment_dir: Path) -> Path:
    """Return the first seed sub-folder inside `experiment_dir`.

    Experiment results are stored as  <experiment_dir>/<seed>/  where <seed>
    is an integer.  The exact seed value can change between runs, so instead
    of hard-coding it we just pick the first (lexicographically) sub-directory
    whose name is a pure integer.
    """
    candidates = sorted(
        [d for d in experiment_dir.iterdir() if d.is_dir() and d.name.isdigit()],
        key=lambda d: int(d.name),
    )
    if not candidates:
        raise FileNotFoundError(
            f"No seed sub-folder found in {experiment_dir}. "
            f"Contents: {[p.name for p in experiment_dir.iterdir()]}"
        )
    return candidates[0]

def gather_csvs(results_root: Path, csv_filename: str) -> pd.DataFrame:
    """Recursively find all `csv_filename` under `results_root` and concatenate."""
    pattern = str(results_root / '**' / csv_filename)
    paths = sorted(glob(pattern, recursive=True))
    print(f"Found {len(paths)} files matching '{csv_filename}'")

    frames = []
    for p in paths:
        try:
            df = pd.read_csv(p)
            # Store the relative path of the CSV for provenance
            rel = str(Path(p).relative_to(results_root))
            df['_source_csv'] = rel
            # Extract the experiment folder (parent directory of the CSV)
            df['_exp_folder'] = str(Path(rel).parent)
            # Overwrite the CSV-internal results_folder with the actual
            # on-disk path (experiments may have been moved/renamed).
            if 'results_folder' in df.columns:
                df['results_folder'] = df['_exp_folder']
            frames.append(df)
        except Exception as e:
            print(f"  ⚠ Skipped {p}: {e}")

    if not frames:
        raise RuntimeError(f"No valid CSVs found for '{csv_filename}' under {results_root}")

    merged = pd.concat(frames, ignore_index=True, sort=False)
    print(f"Merged shape: {merged.shape}")
    return merged


if REBUILD_MERGED_CSV:
    print("Scanning result folders …")
    exp1_all = gather_csvs(RESULTS_ROOT, 'experiment1_pcl_results.csv')
    exp2_all = gather_csvs(RESULTS_ROOT, 'experiment2_accuracy_results.csv')

    # Save to disk
    exp1_all.to_csv(MERGED_EXP1_CSV, index=False)
    exp2_all.to_csv(MERGED_EXP2_CSV, index=False)
    print(f"\nSaved merged CSVs to:\n  {MERGED_EXP1_CSV}\n  {MERGED_EXP2_CSV}")
else:

    print("Loading previously merged CSVs …")    
    print(f"exp1 shape: {exp1_all.shape}, exp2 shape: {exp2_all.shape}")

    exp1_all = pd.read_csv(MERGED_EXP1_CSV)    
    exp2_all = pd.read_csv(MERGED_EXP2_CSV)

In [ ]:
# Quick sanity check
print("=" * 60)
print("Experiment 1 (PCL results) overview")
print("=" * 60)
print(f"Total rows: {len(exp1_all)}")
print(f"Unique source CSVs: {exp1_all['_source_csv'].nunique()}")
print(f"Unique experiment settings: {sorted(exp1_all['experiment_setting'].dropna().unique())}")
print(f"Model types: {sorted(exp1_all['model_type'].dropna().unique())}")
if 'digits_per_class' in exp1_all.columns:
    print(f"digits_per_class values: {sorted(exp1_all['digits_per_class'].dropna().unique())}")

# Build a config key for each experiment
CONFIG_COLS = [
    'experiment_setting', 'seed', 'network_name', 'spur_prob', 'flip_prob',
    'env_noisiness', 'has_watermark', 'uninformative_majority', 'digits_per_class',
]
# Add watermark_bank_size if present (Setting 2)
if 'watermark_bank_size' in exp1_all.columns:
    CONFIG_COLS.append('watermark_bank_size')

# Filter to only cols that exist in both dataframes
config_cols_exp1 = [c for c in CONFIG_COLS if c in exp1_all.columns]
config_cols_exp2 = [c for c in CONFIG_COLS if c in exp2_all.columns]

def make_config_key(row, cols):
    """Create a hashable config string from selected columns."""
    parts = []
    for c in cols:
        v = row.get(c, None)
        if pd.isna(v):
            parts.append(f"{c}=NA")
        else:
            parts.append(f"{c}={v}")
    return ' | '.join(parts)

## 2. Compute Empirical & Theoretical Switch Points

For each experiment configuration:

**Empirical switch** (from Experiment 2):  
The dataset size $N$ at which the dominant accuracy drop switches between features—i.e., where `|acc_drop_A|` crosses `|acc_drop_B|`. We detect the crossover between the two non-Bayes features (e.g. watermark ↔ digit, or color ↔ digit).

**Theoretical switch** (from Experiment 1):  
The MDL-predicted transition $N$ where the best-compressing model type changes between the two non-Bayes features (the purple vertical line). This is computed from the compression envelope built from PCL curves.

In [ ]:
from source.utils.eval_helper import (
    NATS_TO_BITS,
    build_interpolated_threshold_models,
    add_asymptotic_model,
    compute_envelope_indices,
    get_mdl_predicted_quantities,
    find_type_transitions,
)

from source.utils.notebook_helpers import (
    _compute_kp_with_convergence_cutoff, 

)
def compute_theoretical_switch(exp1_df, include_bayes=False, bayes_asymptotic_only=False,
                               kp_abs_delta_threshold=5e-5):
    """
    Given the PCL results for ONE experiment config, compute all MDL-predicted
    dataset sizes at which the dominant feature type switches between the two
    non-Bayes features.

    Parameters
    ----------
    exp1_df : DataFrame
        PCL results for one experiment configuration.
    include_bayes : bool
        If True, include the Bayes-Optimal model in the MDL envelope.
    bayes_asymptotic_only : bool
        Only relevant when include_bayes=True.
        If True, include only the asymptotic Bayes line (K(p) + N·loss);
        threshold/interpolated intermediate Bayes models are skipped.
        If False, all Bayes intermediate models are included as well.
    kp_abs_delta_threshold : float
        Absolute delta threshold for K(p) convergence detection.

    Returns a list of (N, from_type, to_type) for every feature-A↔feature-B
    transition found on the envelope. Returns an empty list if none found.
    """
    df_nats = exp1_df.copy()
    for col in df_nats.columns:
        try:
            df_nats[col] = pd.to_numeric(df_nats[col])
        except (ValueError, TypeError):
            pass

    model_types = df_nats['model_type'].unique()

    # Detect feature keys
    has_watermark_model = any('watermark' in m.lower() for m in model_types)
    has_color_model = any('color' in m.lower() for m in model_types)

    if has_watermark_model:
        FEATURE_A = {'name': 'Watermark', 'model_key': 'watermark',
                     'threshold_type': 'watermark_threshold'}
    elif has_color_model:
        FEATURE_A = {'name': 'Color', 'model_key': 'color',
                     'threshold_type': 'color_threshold'}
    else:
        return []

    FEATURE_B = {'name': 'Digit', 'model_key': 'digit',
                 'threshold_type': 'digit_threshold'}

    # Optionally include Bayes model as a third competing feature type
    FEATURE_BAYES = {'name': 'Bayes', 'model_key': 'bayes',
                     'threshold_type': 'bayes_threshold'}

    # K(p) for each model
    k_p_results_env = {}
    for mt, mdf in df_nats.groupby('model_type'):
        # Skip Bayes unless explicitly included
        if 'bayes' in mt.lower() and not include_bayes:
            continue
        mdf = mdf.sort_values('dataset_size')
        x_m = mdf['dataset_size'].values
        if len(x_m) < 2:
            continue

        asym_row = mdf.loc[mdf['dataset_size'].idxmax()]
        asym_orig_loss = float(asym_row['mean_original_log_loss'])
        asym_orig_acc = float(asym_row.get('mean_original_acc', np.nan))
        asym_test_acc = float(asym_row.get('mean_test_acc', np.nan))
        asym_test_loss = float(asym_row['mean_test_log_loss'])
        asym_gs_acc = float(asym_row.get('mean_grayscale_acc', np.nan))
        asym_gs_ll = float(asym_row.get('mean_grayscale_log_loss', np.nan))
        asym_wm_acc = float(asym_row.get('mean_watermark_only_acc', np.nan)) \
            if 'mean_watermark_only_acc' in mdf.columns else np.nan
        asym_wm_ll = float(asym_row.get('mean_watermark_only_log_loss', np.nan)) \
            if 'mean_watermark_only_log_loss' in mdf.columns else np.nan

        if mt == 'Bayes-Optimal predictor' and 'k_p_nats' in mdf.columns and mdf['k_p_nats'].notna().any():
            k_p = float(mdf['k_p_nats'].iloc[0])
        else:
            y_m = mdf['mean_test_log_loss'].values
            k_p, conv_idx, area_under_curve, asymptotic_test_loss, width_cut = _compute_kp_with_convergence_cutoff(x_m, y_m, abs_delta_threshold=kp_abs_delta_threshold)
    
        k_p_results_env[mt] = {
            'Kp_nats': k_p,  # k_p is already in same units as input df
            'asymptotic_original_loss_nats': asym_orig_loss,
            'asymptotic_original_acc': asym_orig_acc,
            'asymptotic_test_acc': asym_test_acc,
            'asymptotic_test_loss_nats': asym_test_loss,
            'asymptotic_grayscale_acc': asym_gs_acc,
            'asymptotic_grayscale_loss_nats': asym_gs_ll,
            'asymptotic_watermark_only_acc': asym_wm_acc,
            'asymptotic_watermark_only_loss_nats': asym_wm_ll,
            'max_size': x_m[-1],
        }

    if len(k_p_results_env) < 2:
        return []

    # Build threshold models & envelope
    x_min = df_nats['dataset_size'].min()
    df_nats['dataset_size'].max()

    num_interpolated_models = NB_INTERPOLATED_MODELS
    has_wm_only = 'mean_watermark_only_acc' in df_nats.columns

    threshold_models = []
    pvalue_cols_from_df = [
        c for c in df_nats.columns
        if c.startswith('mean_pvalue_') or c.startswith('mean_acc_drop_')
    ]

    # Which features participate in the threshold/interpolated envelope models.
    # Bayes intermediates are only added here when not in asymptotic-only mode.
    features_for_envelope = [FEATURE_A, FEATURE_B]
    if include_bayes and not bayes_asymptotic_only:
        features_for_envelope.append(FEATURE_BAYES)

    for feature in features_for_envelope:
        for cand in [m for m in df_nats['model_type'].unique()
                     if feature['model_key'] in m.lower()]:
            mdf = df_nats[df_nats['model_type'] == cand].sort_values('dataset_size')
            if len(mdf) < 2:
                continue
            tms_interp = build_interpolated_threshold_models(
                mdf, feature['model_key'], feature['name'],
                feature['threshold_type'],
                nb_models=num_interpolated_models, has_watermark_only=has_wm_only,
                interpolation_metric=THRESHOLD_METRIC)
            threshold_models.extend(tms_interp)

    # Add asymptotic models (always added for every included model type).
    for model_name in k_p_results_env:
        if FEATURE_A['model_key'] in model_name.lower():
            ft_name, ft_type = FEATURE_A['name'], FEATURE_A['threshold_type']
        elif FEATURE_B['model_key'] in model_name.lower():
            ft_name, ft_type = FEATURE_B['name'], FEATURE_B['threshold_type']
        elif include_bayes and 'bayes' in model_name.lower():
            ft_name, ft_type = FEATURE_BAYES['name'], FEATURE_BAYES['threshold_type']
        else:
            continue
        asym = add_asymptotic_model(
            k_p_results_env, model_name, ft_name, ft_type,
            pvalue_cols_from_df, df_nats
        )
        threshold_models.append(asym)

    if not threshold_models:
        return []

    # Use the same detection technique as the per-experiment regen plots:
    # sample the lower envelope at 2000 points over [x_min, x_plot_max]
    # then detect type transitions with find_type_transitions().
    x_min_env = max(float(x_min), 1.0)
    x_plot_max = 260_000
    x_linear = np.logspace(np.log10(x_min_env), np.log10(x_plot_max), 2000)

    mdl_pred = get_mdl_predicted_quantities(threshold_models, x_linear)
    transitions = find_type_transitions(mdl_pred['model_type'], x_linear)

    fa_type = FEATURE_A['threshold_type']
    fb_type = FEATURE_B['threshold_type']
    FEATURE_BAYES['threshold_type']

    candidates = []
    for tp in transitions:
        types_involved = {tp['from_type'], tp['to_type']}
        # Collect direct A↔B transitions AND any Bayes-involved transitions (Bayes↔A, Bayes↔B)
        if fa_type in types_involved or fb_type in types_involved:
            candidates.append((tp['N'], tp['from_type'], tp['to_type']))

    return candidates


def compute_empirical_switches(exp2_df, feature_a_attr='watermark', feature_b_attr='digit', min_N=10):
    """
    Given the accuracy results for ONE experiment config, compute ALL dataset
    sizes at which the learned model switches dominant feature reliance.

    We detect every crossover point where |acc_drop_A| and |acc_drop_B| cross.
    Crossovers below `min_N` samples are ignored (too noisy).

    Returns a list of interpolated crossover N values (floats).
    Returns an empty list if no crossovers are found.
    """
    col_a = f'acc_drop_{feature_a_attr}'
    col_b = f'acc_drop_{feature_b_attr}'

    if col_a not in exp2_df.columns or col_b not in exp2_df.columns:
        return []

    # Aggregate across runs: mean acc_drop per dataset_size
    agg = exp2_df.groupby('dataset_size').agg(
        drop_a=(col_a, 'mean'),
        drop_b=(col_b, 'mean'),
    ).reset_index().sort_values('dataset_size')

    sizes = agg['dataset_size'].values.astype(float)
    drop_a = np.abs(agg['drop_a'].values)
    drop_b = np.abs(agg['drop_b'].values)

    # Only keep data points above the minimum dataset-size threshold
    mask = sizes >= min_N
    sizes = sizes[mask]
    drop_a = drop_a[mask]
    drop_b = drop_b[mask]

    if len(sizes) < 2:
        return []

    # Find ALL crossovers: where (drop_a - drop_b) changes sign
    diff = drop_a - drop_b
    sign_changes = np.where(np.diff(np.sign(diff)) != 0)[0]

    if len(sign_changes) == 0:
        return []

    crossovers = []
    for idx in sign_changes:
        x1, x2 = np.log10(sizes[idx]), np.log10(sizes[idx + 1])
        d1, d2 = diff[idx], diff[idx + 1]
        if d2 - d1 == 0:
            continue
        x_cross = x1 - d1 * (x2 - x1) / (d2 - d1)
        crossovers.append(10 ** x_cross)

    return crossovers


def match_closest_pairs(empirical_Ns, theoretical_candidates):
    """
    Match empirical crossover N values with theoretical transition candidates.
    Returns only the SINGLE closest (empirical, theoretical) pair in log-space.

    Parameters
    ----------
    empirical_Ns : list of float
        All empirical crossover dataset sizes.
    theoretical_candidates : list of (N, from_type, to_type)
        All theoretical transition candidates.

    Returns
    -------
    list of (N_empirical, N_theoretical, from_type, to_type)
        At most one element: the closest pair across all combinations.
    """
    if not empirical_Ns or not theoretical_candidates:
        return []

    # Find the single pair with the smallest distance in log-space
    best = None
    best_dist = float('inf')
    for n_emp in empirical_Ns:
        for n_theo, ft, tt in theoretical_candidates:
            dist = abs(np.log10(max(n_emp, 1)) - np.log10(max(n_theo, 1)))
            if dist < best_dist:
                best_dist = dist
                best = (n_emp, n_theo, ft, tt)

    if best is None:
        return []
    return [best]

print("Functions defined.")

In [ ]:
# ── Compute switch points for every experiment folder ────────────────

unique_folders = sorted(
    set(exp1_all['_exp_folder'].unique()) & set(exp2_all['_exp_folder'].unique())
)
print(f"Experiment folders with BOTH exp1 and exp2 CSVs: {len(unique_folders)}")

records = []
n_ok, n_fail = 0, 0

for folder in unique_folders:
    e1 = exp1_all[exp1_all['_exp_folder'] == folder]
    e2 = exp2_all[exp2_all['_exp_folder'] == folder]

    # Detect feature attributes — check for columns with actual non-NaN data
    # (merged DataFrames have ALL acc_drop_* columns; missing settings are NaN)
    drop_cols = [
        c for c in e2.columns
        if c.startswith('acc_drop_') and e2[c].notna().any()
    ]
    feature_attrs = [c.replace('acc_drop_', '') for c in drop_cols]

    # Determine feature_a / feature_b
    if 'watermark' in feature_attrs and 'digit' in feature_attrs:
        fa_attr, fb_attr = 'watermark', 'digit'
    elif 'color' in feature_attrs and 'digit' in feature_attrs:
        fa_attr, fb_attr = 'color', 'digit'
    else:
        # Unknown feature pair — skip
        continue

    # Compute ALL empirical crossovers
    try:
        empirical_Ns = compute_empirical_switches(e2, fa_attr, fb_attr)
    except Exception:
        empirical_Ns = []

    # Per-experiment Bayes skip: mirrors the individual-plot logic
    # (Bayes is uninformative when spur_prob=0 for S1, or env_noisiness=0 for S2)
    _meta0 = e1.iloc[0]
    _setting = int(_meta0.get('experiment_setting', 0))
    _spur = float(_meta0.get('spur_prob', 0) or 0)
    _env_noise = float(_meta0.get('env_noisiness', 0) or 0)
    _skip_bayes_here = (
        SKIP_BAYES_GLOBAL
        or (_setting == 1 and _spur == 0)
        or (_setting == 2 and _env_noise == 0)
    )
    try:
        theory_candidates = compute_theoretical_switch(
            e1,
            include_bayes=INCLUDE_BAYES_IN_ENVELOPE and not _skip_bayes_here,
            bayes_asymptotic_only=BAYES_ASYMPTOTIC_ONLY,
            kp_abs_delta_threshold=KP_ABS_DELTA_THRESHOLD,
        )
    except Exception:
        theory_candidates = []

    # ── Match empirical ↔ theoretical by closest pairs in log-space ──
    matched_pairs = match_closest_pairs(empirical_Ns, theory_candidates)

    # Grab metadata from the first row
    meta = e1.iloc[0]
    base_rec = {
        'folder': folder,
        'experiment_setting': meta.get('experiment_setting', np.nan),
        'network_name': meta.get('network_name', ''),
        'spur_prob': meta.get('spur_prob', np.nan),
        'flip_prob': meta.get('flip_prob', np.nan),
        'env_noisiness': meta.get('env_noisiness', np.nan),
        'watermark_bank_size': meta.get('watermark_bank_size', np.nan),
        'uninformative_majority': meta.get('uninformative_majority', np.nan),
        'digits_per_class': meta.get('digits_per_class', np.nan),
        'seed': meta.get('seed', np.nan),
        'feature_a': fa_attr,
        'feature_b': fb_attr,
    }

    if matched_pairs:
        # One record per matched pair
        for pair_idx, (n_emp, n_theo, ft, tt) in enumerate(matched_pairs):
            # Discard theoretical switch points below the minimum plausible size
            if n_theo < 20:
                n_theo, ft, tt = np.nan, None, None
            rec = {**base_rec,
                   'pair_index': pair_idx,
                   'N_empirical': n_emp,
                   'N_theoretical': n_theo,
                   'transition_from': ft,
                   'transition_to': tt}
            records.append(rec)
            if not np.isnan(n_emp) and not np.isnan(rec['N_theoretical']):
                n_ok += 1
            else:
                n_fail += 1

        # Also emit unmatched empirical crossovers (no theory partner)
        matched_emp = {p[0] for p in matched_pairs}
        for n_emp in empirical_Ns:
            if n_emp not in matched_emp:
                rec = {**base_rec,
                       'pair_index': None,
                       'N_empirical': n_emp,
                       'N_theoretical': np.nan,
                       'transition_from': None,
                       'transition_to': None}
                records.append(rec)
                n_fail += 1

        # Emit unmatched theoretical transitions (no empirical partner)
        matched_theo_Ns = {p[1] for p in matched_pairs}
        for n_theo, ft, tt in theory_candidates:
            if n_theo not in matched_theo_Ns:
                if n_theo < 20:
                    continue
                rec = {**base_rec,
                       'pair_index': None,
                       'N_empirical': np.nan,
                       'N_theoretical': n_theo,
                       'transition_from': ft,
                       'transition_to': tt}
                records.append(rec)
                n_fail += 1
    else:
        # No matches at all — emit separate rows for whatever we have
        if empirical_Ns:
            for n_emp in empirical_Ns:
                rec = {**base_rec,
                       'pair_index': None,
                       'N_empirical': n_emp,
                       'N_theoretical': np.nan,
                       'transition_from': None,
                       'transition_to': None}
                records.append(rec)
                n_fail += 1
        elif theory_candidates:
            for n_theo, ft, tt in theory_candidates:
                if n_theo < 20:
                    continue
                rec = {**base_rec,
                       'pair_index': None,
                       'N_empirical': np.nan,
                       'N_theoretical': n_theo,
                       'transition_from': ft,
                       'transition_to': tt}
                records.append(rec)
                n_fail += 1
        else:
            # Neither empirical nor theoretical — single empty record
            rec = {**base_rec,
                   'pair_index': None,
                   'N_empirical': np.nan,
                   'N_theoretical': np.nan,
                   'transition_from': None,
                   'transition_to': None}
            records.append(rec)
            n_fail += 1

switch_df = pd.DataFrame(records)
print(f"\nDone. {n_ok} matched pairs with both switch points, {n_fail} with at least one missing.")
print(f"Total rows: {len(switch_df)}")
switch_df

In [ ]:
# Show all rows with both switch points
valid = switch_df.dropna(subset=['N_empirical', 'N_theoretical'])
print(f"Configs with both empirical and theoretical switch points: {len(valid)}")
if len(valid) > 0:
    display(valid[[
        'folder', 'experiment_setting', 'flip_prob', 'env_noisiness',
        'watermark_bank_size', 'N_empirical', 'N_theoretical',
        'transition_from', 'transition_to'
    ]].to_string(index=False))
else:
    print("No configs have both switch points — check data.")
    # Show configs that have at least one
    partial = switch_df[
        switch_df['N_empirical'].notna() | switch_df['N_theoretical'].notna()
    ]
    print(f"\nConfigs with at least one switch point: {len(partial)}")
    display(partial[[
        'folder', 'experiment_setting', 'flip_prob', 'env_noisiness',
        'watermark_bank_size', 'N_empirical', 'N_theoretical',
    ]].head(20).to_string(index=False))

In [ ]:
# Quick check: Setting 1 vs Setting 2 status
for s in [1, 2]:
    sub = switch_df[switch_df['experiment_setting'] == s]
    both = sub.dropna(subset=['N_empirical', 'N_theoretical'])
    print(f"Setting {s}: {len(sub)} rows, {len(both)} with both switch points")
    print(f"  feature_a values: {sub['feature_a'].unique()}")
    if len(both) > 0:
        print(f"  N_empirical range: {both['N_empirical'].min():.0f} – {both['N_empirical'].max():.0f}")
        print(f"  N_theoretical range: {both['N_theoretical'].min():.1f} – {both['N_theoretical'].max():.1f}")


## Scatter Plot: Empirical vs. Theoretical Feature Switch Point (Aggregated)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import pearsonr

# ── Publication style ─────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-colorblind')
mpl.rcParams.update({
    'font.size': 12, 'axes.labelsize': 15, 'axes.titlesize': 15,
    'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 12,
    'figure.titlesize': 16, 'lines.linewidth': 2.0, 'lines.markersize': 8,
    'axes.linewidth': 1.2, 'grid.linewidth': 1.5,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'],
})

# ── Filter to valid data ──────────────────────────────────────────────
plot_df = switch_df.dropna(subset=['N_empirical', 'N_theoretical']).copy()
# Apply N_MIN threshold
plot_df = plot_df[(plot_df['N_empirical'] >= N_MIN) & (plot_df['N_theoretical'] >= N_MIN)]

if len(plot_df) == 0:
    print("WARNING: No valid data points for scatter plot.")
    print("Falling back to showing configs with theoretical switch only.")
    plot_df = switch_df.dropna(subset=['N_theoretical']).copy()
    plot_df = plot_df[plot_df['N_theoretical'] >= N_MIN]
    if len(plot_df) == 0:
        raise RuntimeError("No data for scatter plot at all.")

print(f"Plotting {len(plot_df)} configurations")

# ── Compute Pearson correlation (on log scale) ────────────────────────
r, p_value = pearsonr(np.log10(plot_df['N_theoretical']), np.log10(plot_df['N_empirical']))
print(f"Pearson r (log-log): {r:.3f}, p-value: {p_value:.2e}")

# ── Color by experiment setting ───────────────────────────────────────
setting_colors = {1: '#ad5752', 2: '#748fbe'}
setting_labels = {1: 'Scenario A', 2: 'Scenario B'}

# ── Create figure ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

for setting_val, color in setting_colors.items():
    mask = plot_df['experiment_setting'] == setting_val
    sub = plot_df[mask]
    if len(sub) == 0:
        continue
    ax.scatter(
        sub['N_theoretical'], sub['N_empirical'],
        c=color, s=100, edgecolors='k', linewidths=0.6,
        zorder=5, label=setting_labels[setting_val],
    )

# ── Identity line (y = x) ─────────────────────────────────────────────
all_vals = np.concatenate([
    plot_df['N_empirical'].dropna().values,
    plot_df['N_theoretical'].dropna().values,
])
lo = 10 ** (np.floor(np.log10(all_vals.min())) - 0.2)
hi = 10 ** (np.ceil(np.log10(all_vals.max())) + 0.2)
ax.plot([lo, hi], [lo, hi], '--', color='#8B4789', lw=2, alpha=0.7, zorder=1,
        label='$N_{\\mathrm{theory}} = N_{\\mathrm{empirical}}$')

# ── Add Pearson r annotation ──────────────────────────────────────────
ax.text(
    0.95, 0.05, f'$r = {r:.2f}$',
    transform=ax.transAxes, fontsize=14, fontweight='bold',
    ha='right', va='bottom',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.8)
)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)

ax.set_xlabel('regime switch $N$ (Theory)', fontsize=15, fontweight='bold')
ax.set_ylabel('regime switch $N$ (Empirical)', fontsize=15, fontweight='bold')
ax.set_title('Feature Switch: Theory vs. Actual', fontweight='bold', pad=12)

ax.grid(True, which='both', ls='--', alpha=0.5, lw=1.5)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', which='both', right=False, labelsize=16)
ax.tick_params(axis='x', which='both', top=False, labelsize=16)
ax.legend(loc='upper left', frameon=True, shadow=True, fancybox=True, fontsize=12)

fig.tight_layout()

# ── Save ──────────────────────────────────────────────────────────────
out_pdf = 'scatter_empirical_vs_theoretical_switch.pdf'
if SAVE_FIGURES:
    fig.savefig(OUTPUT_DIR /out_pdf, bbox_inches='tight', facecolor='white')
    print(f"Saved: {out_pdf}")

plt.show()

## Scatter Plot: Empirical vs. Theoretical Feature Switch Point (Setting A & B)

In [ ]:
# ── Merged Plot: Effect of flip_prob (config1) & spur_prob (config2) ───

# ── Load config1 data: Setting 2 colored by flip_prob ────────────────
plot_df1 = switch_df.dropna(subset=['N_empirical', 'N_theoretical']).copy()
plot_df1 = _filter_by_folder_prefix(plot_df1, PLOT1_FOLDER_PREFIX, label='Plot 1')
plot_df1['flip_prob'] = pd.to_numeric(plot_df1['flip_prob'], errors='coerce').fillna(0)

# Filter Setting 2: fix env_noisiness and bank size
s2_all = plot_df1[plot_df1['experiment_setting'] == 2].copy()
s2_all = _filter_by_digits_per_class(s2_all, PLOT1_DIGITS_PER_CLASS_S2, label='Plot 1 S2')
_s2_base = s2_all['env_noisiness'] == PLOT1_S2_ENV_NOISINESS

df_config1_s2 = s2_all[_s2_base].copy()
# Apply N_MIN threshold
df_config1_s2 = df_config1_s2[(df_config1_s2['N_empirical'] >= N_MIN) & (df_config1_s2['N_theoretical'] >= N_MIN)]
print(f"Config 1 Setting 2 (flip_prob): {len(df_config1_s2)} points (after N_MIN={N_MIN} filter)")

# ── Load config2 data: Setting 1 colored by spur_prob ────────────────
plot_df2 = switch_df.dropna(subset=['N_empirical', 'N_theoretical']).copy()
plot_df2 = _filter_by_folder_prefix(plot_df2, PLOT2_FOLDER_PREFIX, label='Plot 2')
plot_df2 = _filter_by_digits_per_class(plot_df2, PLOT2_DIGITS_PER_CLASS, label='Plot 2')
plot_df2['flip_prob'] = pd.to_numeric(plot_df2['flip_prob'], errors='coerce').fillna(0)
plot_df2['spur_prob'] = pd.to_numeric(plot_df2['spur_prob'], errors='coerce').fillna(0)

# Filter Setting 1: fix flip_prob
s1_all = plot_df2[plot_df2['experiment_setting'] == 1].copy()
df_config2_s1 = s1_all[s1_all['flip_prob'] == PLOT2_FLIP_PROB_S1].copy()
# Apply N_MIN threshold
df_config2_s1 = df_config2_s1[(df_config2_s1['N_empirical'] >= N_MIN) & (df_config2_s1['N_theoretical'] >= N_MIN)]
print(f"Config 2 Setting 1 (spur_prob): {len(df_config2_s1)} points (after N_MIN={N_MIN} filter)")

# ── Colormaps and norms ──────────────────────────────────────────────
cmap_flip = mpl.colors.LinearSegmentedColormap.from_list('flip_green', ['#FFFFFF', "#759759"])  # flip_prob (Robust feature noise)
cmap_spur = mpl.colors.LinearSegmentedColormap.from_list('spur_red', ['#FFFFFF', "#864340"])  # spur_prob (Spurious feature noise)

# Separate norms for each colorbar
flip_vmin, flip_vmax = df_config1_s2['flip_prob'].min(), df_config1_s2['flip_prob'].max()
norm_flip = mpl.colors.Normalize(vmin=flip_vmin - 0.02, vmax=flip_vmax + 0.02)

spur_vmin, spur_vmax = df_config2_s1['spur_prob'].min(), df_config2_s1['spur_prob'].max()
norm_spur = mpl.colors.Normalize(vmin=spur_vmin - 0.02, vmax=spur_vmax + 0.02)

# ── Figure with stacked colorbars ─────────────────────────────────────
FIGSIZE = (10, 6.5)
CBAR_FRAC = 0.75   # colorbars occupy 90% of figure height
fig, ax = plt.subplots(figsize=FIGSIZE)

scatter_info = []  # collect (scatter_obj, colorbar_label)

# Plot config1 Setting 2 (flip_prob) - circles
if len(df_config1_s2) > 0:
    sc1 = ax.scatter(
        df_config1_s2['N_theoretical'], df_config1_s2['N_empirical'],
        c=df_config1_s2['flip_prob'], cmap=cmap_flip, norm=norm_flip,
        s=100, edgecolors='k', linewidths=0.6, zorder=5, marker='o',
    )
    scatter_info.append((sc1, 'Robust feature noise'))

# Plot config2 Setting 1 (spur_prob) - squares
if len(df_config2_s1) > 0:
    sc2 = ax.scatter(
        df_config2_s1['N_theoretical'], df_config2_s1['N_empirical'],
        c=df_config2_s1['spur_prob'], cmap=cmap_spur, norm=norm_spur,
        s=100, edgecolors='k', linewidths=0.6, zorder=5, marker='o',
    )
    scatter_info.append((sc2, 'Spurious feature noise'))

# Create stacked colorbars — each occupies half of CBAR_FRAC
_n_cbars = len(scatter_info)
_gap = 0.03  # vertical gap between the two colorbars (fraction of fig)
_each_h = (CBAR_FRAC - _gap) / max(_n_cbars, 1)
_y_start = (1 - CBAR_FRAC) / 2  # center vertically

for idx, (sc, lbl) in enumerate(scatter_info):
    _y = _y_start + (_n_cbars - 1 - idx) * (_each_h + _gap)
    cax = fig.add_axes([0.92, _y, 0.02, _each_h])  # [left, bottom, width, height]
    cbar = fig.colorbar(sc, cax=cax)
    cbar.ax.yaxis.set_ticks_position('left')
    cbar.ax.yaxis.set_label_position('right')
    cbar.set_label(lbl, fontsize=13, fontweight='bold', labelpad=10)
    cbar.ax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.2f'))

# Identity line — auto-range from data
_all_n = np.concatenate([
    df_config1_s2[['N_theoretical', 'N_empirical']].values.ravel(),
    df_config2_s1[['N_theoretical', 'N_empirical']].values.ravel(),
])
lo = 10 ** (np.floor(np.log10(_all_n.min()))+0.1)
hi = 10 ** (np.ceil(np.log10(_all_n.max()))-0.1)
ax.plot([lo, hi], [lo, hi], '--', color='#8B4789', lw=2, alpha=0.7, zorder=1,
        label='$N_{\\mathrm{theory}} = N_{\\mathrm{empirical}}$')

lo = 10 ** (np.floor(np.log10(_all_n.min()))-0.1)
hi = 10 ** (np.ceil(np.log10(_all_n.max()))+0.1)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel('regime switch $N$ (Theory)', fontsize=15, fontweight='bold')
ax.set_ylabel('regime switch $N$ (Empirical)', fontsize=15, fontweight='bold')
ax.grid(True, which='both', ls='--', alpha=0.5, lw=1.5)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', which='both', right=False, labelsize=16)
ax.tick_params(axis='x', which='both', top=False, labelsize=16)

# ── Add "Robust" and "Spurious" text annotations near each cluster ───
if len(df_config1_s2) > 0:
    _cx = 10 ** np.log10(df_config1_s2['N_theoretical'].values).mean()
    _cy = 10 ** np.log10(df_config1_s2['N_empirical'].values).mean()
    ax.annotate('Scenario B', xy=(_cx, _cy), fontsize=15, fontweight='bold',
                color=cmap_flip(0.75), ha='center', va='bottom',
                xytext=(45, -45), textcoords='offset points',
                zorder=10)

if len(df_config2_s1) > 0:
    _cx = 10 ** np.log10(df_config2_s1['N_theoretical'].values).mean()
    _cy = 10 ** np.log10(df_config2_s1['N_empirical'].values).mean()
    ax.annotate('Scenario A', xy=(_cx, _cy), fontsize=15, fontweight='bold',
                color=cmap_spur(0.75), ha='center', va='bottom',
                xytext=(-65, 24), textcoords='offset points',
                zorder=10)

# Legend for marker shapes
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], ls='--', color='#8B4789', lw=2,
           label='$N_{\\mathrm{theory}} = N_{\\mathrm{empirical}}$'),
]
ax.legend(
    handles=legend_handles, 
    loc='upper left', 
    bbox_to_anchor=(0.03, 0.97),
    frameon=True, 
    shadow=True, 
    fancybox=True, 
    fontsize=12
)

fig.subplots_adjust(right=0.88)  # make room for colorbars

out_pdf = 'scatter_slope.pdf'
if SAVE_FIGURES:
    fig.savefig(OUTPUT_DIR /out_pdf, bbox_inches='tight', facecolor='white')
    print(f"Saved: {out_pdf}")

plt.show()

In [ ]:

# ── Plot 3: Effect of watermark bank size (Setting 2 only) ────────────
# Fix flip_prob and env_noisiness, vary watermark_bank_size

plot_df = switch_df.dropna(subset=['N_empirical', 'N_theoretical']).copy()
# Apply N_MIN threshold
plot_df = plot_df[(plot_df['N_empirical'] >= N_MIN) & (plot_df['N_theoretical'] >= N_MIN)]
plot_df = _filter_by_folder_prefix(plot_df, PLOT3_FOLDER_PREFIX, label='Plot 3')
plot_df = _filter_by_digits_per_class(plot_df, PLOT3_DIGITS_PER_CLASS, label='Plot 3')
plot_df['flip_prob'] = pd.to_numeric(plot_df['flip_prob'], errors='coerce').fillna(0)
plot_df['env_noisiness'] = pd.to_numeric(plot_df['env_noisiness'], errors='coerce').fillna(0)

# Setting 2 only, fix flip_prob and env_noisiness
mask = (
    (plot_df['experiment_setting'] == 2)
    & (plot_df['flip_prob'] == PLOT3_FLIP_PROB)
    & (plot_df['env_noisiness'] == PLOT3_ENV_NOISE)
)
plot_df = plot_df[mask].copy()

# Ensure watermark_bank_size is numeric
if 'watermark_bank_size' in plot_df.columns:
    plot_df['watermark_bank_size'] = pd.to_numeric(
        plot_df['watermark_bank_size'], errors='coerce'
    )


plot_df = plot_df.dropna(subset=['watermark_bank_size'])
print(f"After filtering: {len(plot_df)} points (Setting 2, flip={PLOT3_FLIP_PROB}, env={PLOT3_ENV_NOISE})")
print(f"  Bank sizes: {sorted(plot_df['watermark_bank_size'].unique())}")

# ── Colormap ──────────────────────────────────────────────────────────
cmap =  mpl.colors.LinearSegmentedColormap.from_list('flip_green', ['#FFFFFF', "#61789f"])  # flip_prob (Robust feature noise)

vals = plot_df['watermark_bank_size'].values
vmin, vmax = vals.min(), vals.max()
norm = mpl.colors.LogNorm(vmin=vmin, vmax=vmax)

# ── Figure (same layout approach as scatter_slope) ────────────────────
FIGSIZE = (10, 6.5)
CBAR_FRAC = 0.70   # colorbar occupies 90% of figure height
fig, ax = plt.subplots(figsize=FIGSIZE)

sc = ax.scatter(
    plot_df['N_theoretical'], plot_df['N_empirical'],
    c=vals, cmap=cmap, norm=norm,
    s=100, edgecolors='k', linewidths=0.6, zorder=5,
)

_y_start = (1 - CBAR_FRAC) / 2
cax = fig.add_axes([0.92, _y_start, 0.02, CBAR_FRAC])
cbar = fig.colorbar(sc, cax=cax)
cbar.ax.yaxis.set_ticks_position('left')
cbar.ax.yaxis.set_label_position('right')

# Show watermark bank size ticks on the colorbar (log-spaced)
_cbar_ticks = np.logspace(np.log10(10), np.log10(300), 5)
_cbar_ticks = np.round(_cbar_ticks).astype(int)
cbar.set_ticks(_cbar_ticks)
cbar.set_ticklabels([str(t) for t in _cbar_ticks])
cbar.set_label('watermark bank size $K$', fontsize=14, fontweight='bold', labelpad=10)

# Identity line — auto-range from data
all_vals = np.concatenate([
    plot_df['N_empirical'].dropna().values,
    plot_df['N_theoretical'].dropna().values,
])
lo = 10 ** (np.floor(np.log10(all_vals.min())))
hi = 10 ** (np.ceil(np.log10(all_vals.max())))
ax.plot([lo, hi], [lo, hi], '--', color='#8B4789', lw=2, alpha=0.7, zorder=1,
        label='$N_{\\mathrm{theory}} = N_{\\mathrm{empirical}}$')


lo = 10 ** (np.floor(np.log10(all_vals.min()))-0.2)
hi = 10 ** (np.ceil(np.log10(all_vals.max()))+0.2)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel('regime switch $N$ (Theory)', fontsize=15, fontweight='bold')
ax.set_ylabel('regime switch $N$ (Empirical)', fontsize=15, fontweight='bold')
ax.grid(True, which='both', ls='--', alpha=0.5, lw=1.5)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', which='both', right=False, labelsize=16)
ax.tick_params(axis='x', which='both', top=False, labelsize=16)

legend_handles = [
    Line2D([0], [0], ls='--', color='#8B4789', lw=2,
           label='$N_{\\mathrm{theory}} = N_{\\mathrm{empirical}}$'),
]
ax.legend(
    handles=legend_handles, 
    loc='upper left', 
    bbox_to_anchor=(0.03, 0.97),
    frameon=True, 
    shadow=True, 
    fancybox=True, 
    fontsize=12
)

fig.subplots_adjust(right=0.88)  # make room for colorbar

out_pdf = 'scatter_complexity.pdf'
if SAVE_FIGURES:
    fig.savefig(OUTPUT_DIR /out_pdf, bbox_inches='tight', facecolor='white')
    print(f"Saved: {out_pdf}")

plt.show()

In [ ]:
# Save the switch-point table for future reference
switch_csv = OUTPUT_DIR / 'feature_switch_points.csv'
if SAVE_FIGURES:
    switch_df.to_csv(switch_csv, index=False)
    print(f"Saved switch-point table to {switch_csv}")
print(f"\nSummary:")
print(f"  Total experiments processed: {len(switch_df)}")
print(f"  With both switch points:     {switch_df.dropna(subset=['N_empirical', 'N_theoretical']).shape[0]}")
print(f"  Empirical only:              {switch_df['N_empirical'].notna().sum() - switch_df.dropna(subset=['N_empirical', 'N_theoretical']).shape[0]}")
print(f"  Theoretical only:            {switch_df['N_theoretical'].notna().sum() - switch_df.dropna(subset=['N_empirical', 'N_theoretical']).shape[0]}")
print(f"  Neither:                     {switch_df[['N_empirical', 'N_theoretical']].isna().all(axis=1).sum()}")

In [ ]:
# ── Folders with only one of the two switch points ────────────────────
has_emp = switch_df['N_empirical'].notna()
has_theo = switch_df['N_theoretical'].notna()
only_one = switch_df[(has_emp ^ has_theo)].copy()

emp_only = only_one[only_one['N_empirical'].notna()]
theo_only = only_one[only_one['N_theoretical'].notna()]

print(f"Folders with empirical switch only ({len(emp_only)}):")
for _, row in emp_only.iterrows():
    print(f"  {row['folder']}  (N_empirical={row['N_empirical']:.1f})")

print(f"\nFolders with theoretical switch only ({len(theo_only)}):")
for _, row in theo_only.iterrows():
    print(f"  {row['folder']}  (N_theoretical={row['N_theoretical']:.1f})")

## Figure 2: side-by-side Comparison: Setting 1 vs Setting 2

Select one experiment folder per setting. Produces a **3 rows × 2 columns** figure:
- **Row 1 — Compression envelope** (intermediate + asymptotic lines)
- **Row 2 — Accuracy** on different evaluation datasets
- **Row 3 — Feature reliance**: accuracy drop under permutation

Toggle `SHOW_MDL_PREDICTED` to overlay / hide the MDL-predicted dashed lines on rows 2 and 3.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ██  CONFIGURATION — Side-by-side comparison figure  ██
# ═══════════════════════════════════════════════════════════════════════

# ── Experiment folders (one per column) ──────────────────────────────
SIDEBYSIDE_FOLDER_S1 = _resolve_seed_folder(
    RESULTS_ROOT / 'config2_setting1_MLP_spur0.27_flip0.0_env_noise0.0_uninfFalse'
)
SIDEBYSIDE_FOLDER_S2 = _resolve_seed_folder(
    RESULTS_ROOT / 'config1_setting2_MLP_spur0.5_flip0.15_env_noise0.0_bank50'
)
# ── Toggle: show MDL-predicted dashed lines in Accuracy & Acc-Drop rows
SHOW_MDL_PREDICTED = True

# ── Skip Bayes curves ────────────────────────────────────────────────
SIDEBYSIDE_SKIP_BAYES = True
NB_INTERPOLATED_MODELS = 50       # Loss-interpolated intermediate models

# ── Legend label renaming ────────────────────────────────────────────
# Map any internal name → display name.  Keys can be model names (row 0),
# accuracy legend entries (row 1), or permutation attribute names (row 2).
# Any name not in the dict keeps its original value.
LEGEND_RENAME = {
    # Row 0 – envelope asymptotic lines (keys = model names from PCL CSV)
    # e.g. 'simple_threshold': 'Color',
    # Row 1 – accuracy curves
    # 'Train': 'Train',
    # 'Validation': 'Val',
    'Digit-based': 'Digit feature',
    'Color-based': 'Color feature',
    'Watermark-only': 'Watermark feature',
    # 'Learned': 'Learned',
    'MDL predicted': 'Predicted',
    'Digit-Only': 'Digit-only',
    # Row 2 – feature-reliance (keys = attr.capitalize())
    # 'Color': 'Color',
    # 'Watermark': 'Watermark',
}

# ═══════════════════════════════════════════════════════════════════════

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

from source.utils.plotting import (
    COLOR_MAP, COLOR_TRAIN_ACC, COLOR_VAL_ACC,
    apply_publication_style,
    _auto_detect_features, _compute_k_p_results, _aggregate_exp2,
)
from source.utils.eval_helper import (
    NATS_TO_BITS,
    build_threshold_lines,

)

apply_publication_style()

# ── Load data for both settings ──────────────────────────────────────
folders = {
    'Scenario A': SIDEBYSIDE_FOLDER_S1,
    'Scenario B': SIDEBYSIDE_FOLDER_S2,
}

loaded = {}
for label, folder in folders.items():
    pcl_csv = folder / 'experiment1_pcl_results.csv'
    acc_csv = folder / 'experiment2_accuracy_results.csv'
    assert pcl_csv.exists(), f"Missing {pcl_csv}"
    assert acc_csv.exists(), f"Missing {acc_csv}"
    pcl_df = pd.read_csv(pcl_csv)
    acc_df = pd.read_csv(acc_csv)
    for col in pcl_df.columns:
        try:
            pcl_df[col] = pd.to_numeric(pcl_df[col])
        except (ValueError, TypeError):
            pass
    meta = pcl_df.iloc[0]
    loaded[label] = {
        'pcl_df': pcl_df,
        'acc_df': acc_df,
        'experiment_setting': int(meta.get('experiment_setting', 1)),
        'spur_prob': float(meta.get('spur_prob', 0)),
        'flip_prob': float(meta.get('flip_prob', 0)),
        'env_noisiness': float(meta.get('env_noisiness', 0)),
        'watermark_bank_size': int(meta.get('watermark_bank_size', 2))
            if 'watermark_bank_size' in pcl_df.columns else 2,
    }
    print(f"Loaded {label} from {folder}")


# ── Build figure: 3 rows × 2 columns ─────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 14))

for col_idx, (label, data) in enumerate(loaded.items()):
    pcl_df = data['pcl_df']
    acc_df = data['acc_df']
    exp_set = data['experiment_setting']
    skip_bayes = SIDEBYSIDE_SKIP_BAYES

    # Feature detection
    model_types_in_pcl = pcl_df['model_type'].unique()
    FEATURE_A, FEATURE_B, SEC_COL, SEC_LBL = _auto_detect_features(model_types_in_pcl)
    BAYES_THRESHOLD_TYPE = 'bayes_threshold'

    # Permutation attributes
    perm_attrs = [c.replace('pvalue_', '') for c in acc_df.columns if c.startswith('pvalue_')]

    # Aggregate exp2
    summary_df = _aggregate_exp2(acc_df, exp_set, perm_attrs)

    # K(p)
    k_p_results_env = _compute_k_p_results(pcl_df, skip_bayes,
                                            kp_abs_delta_threshold=KP_ABS_DELTA_THRESHOLD)
    if len(k_p_results_env) < 2:
        print(f"  WARNING: {label} — fewer than 2 models, skipping column")
        continue

    # Build threshold / envelope models
    x_min = pcl_df['dataset_size'].min()
    x_max = pcl_df['dataset_size'].max()
    x_plot_max = 249_999
    x_linear = np.logspace(np.log10(max(x_min, 1)), np.log10(x_max), 4000)
    has_wm_only = 'mean_watermark_only_acc' in pcl_df.columns

    threshold_models = []
    threshold_lines_list = []
    for feature in [FEATURE_B, FEATURE_A]:
        for cand in [m for m in model_types_in_pcl if feature['model_key'] in m.lower()]:
            mdf = pcl_df[pcl_df['model_type'] == cand].sort_values('dataset_size')
            if len(mdf) < 2:
                continue
            tms_interp = build_interpolated_threshold_models(
                mdf, feature['model_key'], feature['name'], feature['threshold_type'],
                nb_models=NB_INTERPOLATED_MODELS, has_watermark_only=has_wm_only,
                interpolation_metric=THRESHOLD_METRIC,
            )
            threshold_models.extend(tms_interp)
            threshold_lines_list.extend(build_threshold_lines(tms_interp, x_linear))

    pvalue_cols = [c for c in pcl_df.columns
                   if c.startswith('mean_pvalue_') or c.startswith('mean_acc_drop_')]

    # Track which indices are asymptotic models (for labelling in envelope)
    asymptotic_indices = {}  # index → model_name
    for model_name in k_p_results_env:
        if FEATURE_A['model_key'] in model_name.lower():
            ft_name, ft_type = FEATURE_A['name'], FEATURE_A['threshold_type']
        elif FEATURE_B['model_key'] in model_name.lower():
            ft_name, ft_type = FEATURE_B['name'], FEATURE_B['threshold_type']
        elif 'bayes' in model_name.lower():
            if skip_bayes:
                continue
            ft_name, ft_type = 'Bayes', BAYES_THRESHOLD_TYPE
        else:
            continue
        asym = add_asymptotic_model(k_p_results_env, model_name, ft_name, ft_type,
                                    pvalue_cols, pcl_df)
        asym_idx = len(threshold_models)
        asymptotic_indices[asym_idx] = model_name
        threshold_models.append(asym)
        threshold_lines_list.append(asym['k_p'] + x_linear * asym['slope'])

    if not threshold_models:
        print(f"  WARNING: {label} — no threshold models, skipping column")
        continue

    envelope_indices = compute_envelope_indices(threshold_models, x_min, x_max)
    mdl_pred = get_mdl_predicted_quantities(threshold_models, x_linear)
    transitions = find_type_transitions(mdl_pred['model_type'], x_linear)

    # Exp2 arrays
    exp2_sizes = summary_df['dataset_size'].values
    exp2_val_acc = summary_df['val_acc_mean'].values
    exp2_train_acc = summary_df['train_acc_mean'].values if 'train_acc_mean' in summary_df.columns else None
    exp2_sec_acc = summary_df[SEC_COL].values if SEC_COL in summary_df.columns else None
    exp2_wm_acc = (summary_df['watermark_only_acc_mean'].values
                   if 'watermark_only_acc_mean' in summary_df.columns else None)
    exp2_color_only_acc = (summary_df['color_only_acc_mean'].values
                           if 'color_only_acc_mean' in summary_df.columns else None)
    exp2_digit_only_acc = (summary_df['digit_only_acc_mean'].values
                           if 'digit_only_acc_mean' in summary_df.columns else None)

    # Colors
    asym_model_colors = {}
    for mn, res in k_p_results_env.items():
        if FEATURE_B['model_key'] in mn.lower():
            asym_model_colors[FEATURE_B['threshold_type']] = COLOR_MAP.get(mn, 'tab:blue')
        elif FEATURE_A['model_key'] in mn.lower():
            asym_model_colors[FEATURE_A['threshold_type']] = COLOR_MAP.get(mn, 'tab:orange')
        elif 'bayes' in mn.lower():
            asym_model_colors[BAYES_THRESHOLD_TYPE] = COLOR_MAP.get(mn, 'tab:green')
    colormaps_by_type = {}
    for mtype, tc in asym_model_colors.items():
        colormaps_by_type[mtype] = LinearSegmentedColormap.from_list(
            f'{mtype}_g', [(0.0, '#C2C2C2'), (1.0, tc)], N=256)
    # Dynamically compute norm based on threshold values and metric type
    _thresholds = [m['threshold'] for m in threshold_models if 'threshold' in m]
    _t_min, _t_max = (min(_thresholds), max(_thresholds)) if _thresholds else (0.5, 1.0)
    if 'k(p)' in THRESHOLD_METRIC.lower() or 'k_p' in THRESHOLD_METRIC.lower():
        norm_all = mpl.colors.LogNorm(vmin=max(_t_min, 1e-6), vmax=_t_max)
    else:
        norm_all = mpl.colors.Normalize(vmin=_t_min, vmax=_t_max)

    color_secondary = asym_model_colors.get(FEATURE_B['threshold_type'], '#2ca02c')
    color_feature_a = asym_model_colors.get(FEATURE_A['threshold_type'], '#1f77b4')

    # ── Row 0: Compression envelope ──────────────────────────────────
    ax_env = axes[0, col_idx]

    # Determine envelope order so that later-envelope models are drawn on top.
    # Off-envelope models get position -1 so they are drawn first (behind).
    envelope_order = []
    for i_env in range(len(threshold_models)):
        if i_env in envelope_indices:
            line_at_x = threshold_models[i_env]['k_p'] + x_linear * threshold_models[i_env]['slope']
            all_lines = np.array([m['k_p'] + x_linear * m['slope'] for m in threshold_models])
            is_best = np.argmin(all_lines, axis=0) == i_env
            first_best = np.argmax(is_best) if is_best.any() else 0
            envelope_order.append((first_best, i_env))
        else:
            envelope_order.append((-1, i_env))

    # Sort: off-envelope first, then on-envelope left-to-right (latest on top)
    envelope_order.sort(key=lambda t: (t[0] >= 0, t[0]))

    # Build per-model zorder: base ordering from envelope sort,
    # then bump each asymptotic line above all others of its model class.
    zorder_map = {}
    for zorder_idx, (env_pos, i_env) in enumerate(envelope_order):
        zorder_map[i_env] = 2 + zorder_idx
    for i_asym in asymptotic_indices:
        mtype_asym = threshold_models[i_asym]['model_type']
        max_class_z = max(
            zorder_map[j] for _, j in envelope_order
            if threshold_models[j].get('model_type') == mtype_asym
        )
        zorder_map[i_asym] = max_class_z + 0.5

    for zorder_idx, (env_pos, i_env) in enumerate(envelope_order):
        model = threshold_models[i_env]
        line_vals = threshold_lines_list[i_env]
        on_envelope = i_env in envelope_indices
        is_asymptotic = i_env in asymptotic_indices
        mtype = model['model_type']
        z = zorder_map[i_env]
        if mtype not in colormaps_by_type:
            continue

        if is_asymptotic:
            lc = COLOR_MAP.get(asymptotic_indices[i_env], 'tab:purple')
            _lbl = LEGEND_RENAME.get(asymptotic_indices[i_env], asymptotic_indices[i_env])
            ax_env.plot(x_linear, line_vals * NATS_TO_BITS, '-', lw=2.5, alpha=0.85,
                        color=lc, zorder=z, label=_lbl)
        else:
            cmap_env = colormaps_by_type[mtype]
            color_env = cmap_env(norm_all(model['threshold']))
            lw = 2
            alpha = 0.85
            ax_env.plot(x_linear, line_vals * NATS_TO_BITS, '-', lw=lw, alpha=alpha,
                        color=color_env, zorder=z)

    for tp in transitions:
        if tp['N'] >= 20:
            ax_env.axvline(x=tp['N'], color='black', ls='-', alpha=0.65, lw=2.0,
                           zorder=2 + len(envelope_order) + 1)

    ax_env.set_xscale('log')
    ax_env.set_yscale('log')
    ax_env.set_xlim(x_min, x_plot_max)
    ax_env.set_xlabel('Dataset size ($N$)')
    if col_idx == 0:
        ax_env.set_ylabel('Total description length (bits)')
    ax_env.set_title(r'Compression Envelope',
                     fontweight='bold', pad=10)
    ax_env.text(0.5, 1.12, label, transform=ax_env.transAxes,
                ha='center', va='bottom', fontsize=25, fontweight='bold')
    ax_env.grid(True, which='both', ls='--', alpha=0.0, lw=1.0)
    env_legend_loc = 'upper left'
    handles_env, labels_env = ax_env.get_legend_handles_labels()
    if col_idx == 0:
        handles_env, labels_env = handles_env[::-1], labels_env[::-1]
    ax_env.legend(handles_env, labels_env, loc=env_legend_loc,
                  bbox_to_anchor=(0.015, 0.97), fontsize=10,
                  frameon=True, shadow=True, fancybox=True)
    ax_env.spines['top'].set_visible(False)
    ax_env.spines['right'].set_visible(False)

    # ── Row 1: Accuracy ──────────────────────────────────────────────
    ax_acc = axes[1, col_idx]
    ax_acc.plot(exp2_sizes, exp2_val_acc, '-', color=COLOR_VAL_ACC, lw=2.5, alpha=0.85)
    if exp2_train_acc is not None:
        ax_acc.plot(exp2_sizes, exp2_train_acc, '-', color=COLOR_TRAIN_ACC, lw=2.5, alpha=0.85)
    # Digit-only: use digit_only_acc when available (S1), fall back to grayscale (S2)
    _digit_only_line = exp2_digit_only_acc if exp2_digit_only_acc is not None else exp2_sec_acc
    if _digit_only_line is not None:
        ax_acc.plot(exp2_sizes, _digit_only_line, '-', color=color_secondary, lw=2.5, alpha=0.85)
    if exp2_wm_acc is not None and exp_set == 2:
        ax_acc.plot(exp2_sizes, exp2_wm_acc, '-', color=color_feature_a, lw=2.5, alpha=0.85)
    # Color-only test accuracy (Setting 1 only)
    if exp2_color_only_acc is not None and exp_set == 1:
        ax_acc.plot(exp2_sizes, exp2_color_only_acc, '-',
                    color=COLOR_MAP.get('color', '#ad5752'), lw=2.5, alpha=0.85)

    if SHOW_MDL_PREDICTED:
        mdl_at_sizes = get_mdl_predicted_quantities(threshold_models,
                                                     np.asarray(exp2_sizes, dtype=float))
        # Digit-only theory: use digit_only if available, fall back to grayscale
        _mdl_digit_line = mdl_at_sizes['digit_only_accuracy']
        if np.all(np.isnan(_mdl_digit_line)):
            _mdl_digit_line = mdl_at_sizes['grayscale_accuracy']
        ax_acc.plot(exp2_sizes, _mdl_digit_line, '--',
                    color=color_secondary, lw=2.0, alpha=0.7)
        # Color-only theory (Setting 1)
        if exp_set == 1 and not np.all(np.isnan(mdl_at_sizes['color_only_accuracy'])):
            ax_acc.plot(exp2_sizes, mdl_at_sizes['color_only_accuracy'], '--',
                        color=COLOR_MAP.get('color', '#ad5752'), lw=2.0, alpha=0.7)
        if (not np.all(np.isnan(mdl_at_sizes['watermark_only_accuracy']))) and exp_set == 2:
            ax_acc.plot(exp2_sizes, mdl_at_sizes['watermark_only_accuracy'], '--',
                        color=color_feature_a, lw=2.0, alpha=0.7)

    for tp in transitions:
        if tp['N'] >= 20:
            ax_acc.axvline(x=tp['N'], color='black', ls='-', alpha=0.65, lw=2.0)

    ax_acc.set_xscale('log')
    ax_acc.set_xlim(x_min, x_plot_max)
    ax_acc.set_ylim(0.45, 1.02)
    if col_idx == 0:
        ax_acc.set_ylabel('Accuracy')
    ax_acc.set_title('Accuracy (Theory vs. Actual)', fontweight='bold', pad=10)
    ax_acc.grid(True, which='both', ls='--', alpha=0.5, lw=1.0)
    ax_acc.spines['top'].set_visible(False)
    ax_acc.spines['right'].set_visible(False)

    # Legend for accuracy row
    if col_idx == 0:
        style_handles = [
            Line2D([0], [0], color='gray', lw=2.5, ls='-',
                   label=LEGEND_RENAME.get('Learned', 'Learned')),
            Line2D([0], [0], color='gray', lw=2.5, ls='--',
                   label=LEGEND_RENAME.get('MDL predicted', 'MDL predicted')),
        ]
        ax_acc.legend(handles=style_handles, title='Model',
                      loc='center right', frameon=True, shadow=True, fancybox=True,
                      fontsize=10, title_fontsize=10)
    else:
        ds_handles = [
            Line2D([0], [0], color=COLOR_TRAIN_ACC, lw=2.5,
                   label=LEGEND_RENAME.get('Train', 'Train')),
            Line2D([0], [0], color=COLOR_VAL_ACC, lw=2.5,
                   label=LEGEND_RENAME.get('Validation', 'Validation')),
            Line2D([0], [0], color='none', lw=0, label=' '),  # spacer
            Line2D([0], [0], color=color_secondary, lw=2.5,
                   label=LEGEND_RENAME.get('Digit-Only', 'Digit-Only')),
            Line2D([0], [0], color=COLOR_MAP.get('color', '#ad5752'), lw=2.5,
                   label=LEGEND_RENAME.get('Color-Only', 'Color-Only')),
        ]
        if exp2_wm_acc is not None and exp_set == 2:
            ds_handles.append(Line2D([0], [0], color=color_feature_a, lw=2.5,
                                     label="Watermark-only"))
        ax_acc.legend(handles=ds_handles, title='Dataset',
                      bbox_to_anchor=(0.02, 0.65),
                      loc='center left', frameon=True, shadow=True, fancybox=True,
                      fontsize=10, title_fontsize=10)

    # ── Row 2: Accuracy drop ─────────────────────────────────────────
    ax_drop = axes[2, col_idx]
    for attr in perm_attrs:
        drop_col = f'acc_drop_{attr}_mean'
        drop_std_col = f'acc_drop_{attr}_std'
        if drop_col not in summary_df.columns:
            continue
        drops = summary_df[drop_col].values
        if np.all(np.isnan(drops)):
            continue
        lc = COLOR_MAP.get(attr, 'tab:gray')
        _drop_lbl = LEGEND_RENAME.get(attr.capitalize(), attr.capitalize())
        ax_drop.plot(exp2_sizes, drops, '-', color=lc, lw=2.5, alpha=0.85, label=_drop_lbl)
        if drop_std_col in summary_df.columns:
            stds = summary_df[drop_std_col].values
            ax_drop.fill_between(exp2_sizes, drops - stds, drops + stds,
                                 color=lc, alpha=0.15)

    for tp in transitions:
        if tp['N'] >= 20:
            ax_drop.axvline(x=tp['N'], color='black', ls='-', alpha=0.65, lw=2.0)

    ax_drop.axhline(0, color='gray', ls='-', lw=1, alpha=0.5)
    ax_drop.set_xscale('log')
    ax_drop.set_xlim(x_min, x_plot_max)
    ax_drop.set_ylim(bottom=-0.05)
    ax_drop.grid(True, which='both', ls='--', alpha=0.5, lw=1.0)
    ax_drop.set_xlabel('Dataset size ($N$)')
    if col_idx == 0:
        ax_drop.set_ylabel('Accuracy drop')
    ax_drop.set_title('Feature Reliance', fontweight='bold', pad=10)
    ax_drop.spines['top'].set_visible(False)
    ax_drop.spines['right'].set_visible(False)
    handles_drop, labels_drop = ax_drop.get_legend_handles_labels()
    if col_idx == 0:
        handles_drop, labels_drop = handles_drop[::-1], labels_drop[::-1]
    ax_drop.legend(handles_drop, labels_drop, loc='upper left', fontsize=10,
                   bbox_to_anchor=(0.015, 0.97), frameon=True, shadow=True,
                   fancybox=True)

fig.tight_layout(h_pad=3.5, w_pad=3.0)

# ── Add vertical separator line between the two columns ──────────────
"""fig.add_artist(Line2D([0.51, 0.51], [0.01, 0.95], transform=fig.transFigure,
                      color='black', linewidth=2, linestyle='-', alpha=0.6))
"""
for row_idx, row_lbl in enumerate(['a)', 'b)', 'c)']):
    axes[row_idx, 0].text(-0.05, 1.15, row_lbl,
                          transform=axes[row_idx, 0].transAxes,
                          fontsize=22, fontweight='normal', va='top', ha='right')
# ── Save ──────────────────────────────────────────────────────────────
out_pdf = 'qualitative_study.pdf'

if True:
    fig.savefig(OUTPUT_DIR /out_pdf, bbox_inches='tight', facecolor='white')
    print(f"Saved: {out_pdf}")

plt.show()

## 5. Regenerate per-experiment summary plots

Iterate over every experiment subfolder under `RESULTS_ROOT`, load the two CSVs, recompute the 2×3 summary figure (PCL curves, compression, accuracy, p-values, accuracy drop), and overwrite the existing `experiment_summary_plots.{png,pdf}`.

In [ ]:
from glob import glob
from source.utils.plotting import apply_publication_style, plot_experiment_summary
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ── Find all experiment leaf folders (contain both CSVs) ─────────────
exp1_paths = sorted(glob(str(RESULTS_ROOT / '**' / 'experiment1_pcl_results.csv'), recursive=True))
print(f"Found {len(exp1_paths)} experiment folders with PCL results")

# ── Plot configuration ───────────────────────────────────────────────
SKIP_BAYES_PLOTS_REGEN = True
INCLUDE_BAYES_INTERMEDIATES_REGEN = False
SHOW_TRANSITION_LINES_REGEN = True
SHOW_NON_ENVELOPE_LINES = True

# Apply publication-ready Matplotlib style (once)
apply_publication_style()

n_ok_regen, n_fail_regen = 0, 0

for exp1_csv_path in exp1_paths:
    print(f"Parsing file: {exp1_csv_path}")

    csv_dir = Path(exp1_csv_path).parent
    exp2_csv_path = csv_dir / 'experiment2_accuracy_results.csv'
    if not exp2_csv_path.exists():
        print(f"  SKIP (no exp2 CSV): {csv_dir}")
        n_fail_regen += 1
        continue

    try:
        pcl_results_df = pd.read_csv(exp1_csv_path)
        results_df = pd.read_csv(exp2_csv_path)
    except Exception as exc:
        print(f"  SKIP (read error): {csv_dir} — {exc}")
        n_fail_regen += 1
        continue

    # ── Extract metadata from CSVs ────────────────────────────────────
    _row0 = pcl_results_df.iloc[0]
    _experiment_setting = int(_row0.get('experiment_setting', 1))
    _spur_prob = float(_row0.get('spur_prob', 0))
    _flip_prob = float(_row0.get('flip_prob', 0))
    _env_noisiness = float(_row0.get('env_noisiness', 0))
    _watermark_bank_size = int(_row0.get('watermark_bank_size', 2)) if 'watermark_bank_size' in pcl_results_df.columns else 2
    _uninformative_majority = bool(_row0.get('uninformative_majority', False)) if 'uninformative_majority' in pcl_results_df.columns else False

    _skip_bayes = SKIP_BAYES_PLOTS_REGEN or SKIP_BAYES_GLOBAL

    fig = plot_experiment_summary(
        pcl_results_df,
        results_df,
        experiment_setting=_experiment_setting,
        spur_prob=_spur_prob,
        flip_prob=_flip_prob,
        env_noisiness=_env_noisiness,
        watermark_bank_size=_watermark_bank_size,
        uninformative_majority=_uninformative_majority,
        skip_bayes=_skip_bayes,
        include_bayes_intermediates=INCLUDE_BAYES_INTERMEDIATES_REGEN,
        show_transition_lines=SHOW_TRANSITION_LINES_REGEN,
        show_non_envelope_lines=SHOW_NON_ENVELOPE_LINES,
        nb_threshold_models=NB_THRESHOLD_MODELS,
        nb_interpolated_models=NB_INTERPOLATED_MODELS,
        threshold_metric=THRESHOLD_METRIC,
        kp_abs_delta_threshold=KP_ABS_DELTA_THRESHOLD,
        save_dir=csv_dir,
    )

    if fig is None:
        n_fail_regen += 1
        continue

    #plt.show()
    plt.close(fig)
    n_ok_regen += 1

print(f"\nDone. Regenerated {n_ok_regen} plots, {n_fail_regen} skipped.")